In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

## Data split

The dataset will be divided into:

- 70% training
- 15% validation
- 15% held-out test

Splitting will occur at the patient level to prevent samples from the same
patient appearing in multiple subsets.

Stage distribution will be preserved through stratification where possible.

The held-out test set will not be used for model selection or hyperparameter tuning.

In [ ]:
def create_patient_split(
    clinical,
    patient_col="patient_id",
    stage_col="stage_binary",
    random_state=42
):
    """
    Create 70/15/15 train-validation-test splits at the patient level.
    """

    # Confirm each patient has only one stage label
    stage_per_patient = (
        clinical.groupby(patient_col)[stage_col]
        .nunique()
    )

    if (stage_per_patient > 1).any():
        raise ValueError("Some patients have conflicting stage labels.")

    # One row per patient
    patients = (
        clinical[[patient_col, stage_col]]
        .drop_duplicates(subset=patient_col)
        .copy()
    )

    # 70% train, 30% temporary
    train_patients, temp_patients = train_test_split(
        patients,
        test_size=0.30,
        stratify=patients[stage_col],
        random_state=random_state
    )

    # Split remaining 30% equally:
    # 15% validation, 15% test
    val_patients, test_patients = train_test_split(
        temp_patients,
        test_size=0.50,
        stratify=temp_patients[stage_col],
        random_state=random_state
    )

    return train_patients, val_patients, test_patients

In [ ]:
def summarize_split(name, df, stage_col="stage_binary"):
    print(f"{name}")
    print(f"Patients: {len(df)}")
    print(df[stage_col].value_counts())
    print(df[stage_col].value_counts(normalize=True).round(3))
    print()

In [ ]:
train_patients, val_patients, test_patients = create_patient_split(clinical)

summarize_split("TRAIN", train_patients)
summarize_split("VALIDATION", val_patients)
summarize_split("TEST", test_patients)

## Leakage prevention

The following steps must be performed using training data only:

- feature filtering based on prevalence
- feature selection
- scaling or normalization requiring fitted parameters
- hyperparameter selection

Validation data will be used for model selection.

The test set will remain untouched until the final model configurations are fixed.

## Genomic feature preprocessing

Final preprocessing will be determined after inspecting the files supplied by the data lead.

Items to determine:

### Mutations
- Binary gene-level encoding
- Minimum alteration-frequency threshold
- Gene-panel coverage handling

### CNAs
- Determine original CNA encoding
- Decide whether to preserve deletion/amplification categories or use binary altered/not altered
- Remove constant features

### Combined
- Concatenate mutation and CNA matrices after preprocessing
- Prefix feature names to distinguish alteration types